# 多環境部署 — 從開發到生產

**目標**:在不同環境間切換設定、比較環境差異、理解部署流程

**前置條件**:
- 已完成所有前面的範例
- 理解基本的部署概念


In [ ]:
from llm_framework.config import load_config
import os


## 方法一:直接指定環境


In [ ]:
# 載入開發環境
dev_config = load_config("dev")
print(f"環境: {dev_config.env}")
print(f"模型: {dev_config.llm.default_model}")
print(f"溫度: {dev_config.llm.temperature}")
print(f"MLflow 追蹤: {dev_config.mlflow.tracking_enabled}")


## 方法二:使用環境變數


In [ ]:
# 透過環境變數切換到生產環境
os.environ["LLM_ENV"] = "prod"
prod_config = load_config()
print(f"環境: {prod_config.env}")
print(f"模型: {prod_config.llm.default_model}")
print(f"溫度: {prod_config.llm.temperature}")
print(f"MLflow 追蹤: {prod_config.mlflow.tracking_enabled}")


## 比較各環境差異


In [ ]:
envs = ["dev", "staging", "prod"]
print(f"{'環境':<10} {'模型':<20} {'溫度':<8} {'追蹤':<8}")
print("-" * 50)

for env in envs:
    config = load_config(env)
    print(f"{env:<10} {config.llm.default_model:<20} {config.llm.temperature:<8.1f} {str(config.mlflow.tracking_enabled):<8}")


## 生產環境的輕量安裝

開發環境和生產環境使用不同的依賴:

- **開發環境**: `uv sync --group dev` (完整 MLflow 含 UI)
- **生產環境**: `uv sync --group prod` (僅 mlflow-tracing,無 UI)

這樣可以減少生產環境的容器大小和啟動時間。


## Docker 部署範例

生產環境部署流程。


In [ ]:
dockerfile_content = '''
FROM python:3.12-slim

# 安裝 uv
COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv

WORKDIR /app
COPY . .

# 安裝生產依賴(輕量版)
RUN uv sync --group prod --frozen

# 設定環境
ENV LLM_ENV=prod

# 啟動應用
CMD ["uv", "run", "python", "-m", "llm_framework.api.server"]
'''

print("Dockerfile 範例:")
print(dockerfile_content)


## 總結

恭喜你完成了所有範例!

你已經學會了:
- 使用 LLMClient 呼叫 LLM
- 建立和測試 Workflow 節點
- 組裝完整的多節點 Workflow
- 執行自動化評估
- 在不同環境間切換和部署

**接下來**:
- 探索 `llm_framework/` 原始碼
- 建立自己的專案和 Workflow
- 調整設定檔以符合需求
- 部署到生產環境

祝你在 LLM 開發之路上順利!
